# 06 — Scenario Generation (Morphological)

Generate future scenarios from **CIB-consistent manifestation configurations** (morphological analysis):
1. Load scenario seeds from consistency analysis (NB05b)
2. Build driver assumptions from each seed's manifestation configuration
3. Generate RAG-grounded narratives per scenario

- Input: `consistency_state.json` + `morphbox_state.json` + `cib_state.json` + `merge_state.json`
- Output: `scenario_state.json`

In [ ]:
import sys, os, json
import numpy as np
sys.path.insert(0, os.path.abspath(".."))

from src.llm import safe_chat_json, embed
from src.rag import get_collection
from src.models.drivers import TechDriver
from src.models.scenarios import Scenario, DriverAssumption, ScenarioType
from src.models.morphological import DriverManifestation, MorphologicalBox
from src import prompts

collection = get_collection()

with open("../data/outputs/merge_state.json") as f:
    drivers = [TechDriver(**d) for d in json.load(f)["unified_drivers"]]

with open("../data/outputs/cib_state.json") as f:
    cib = json.load(f)

with open("../data/outputs/morphbox_state.json") as f:
    morphbox_raw = json.load(f)

with open("../data/outputs/consistency_state.json") as f:
    consistency = json.load(f)

morph_box = MorphologicalBox(
    drivers=morphbox_raw["drivers"],
    manifestations=morphbox_raw["manifestations"],
    all_manifestations=[DriverManifestation(**m) for m in morphbox_raw["all_manifestations"]],
)

cib_matrix = np.array(cib["matrix"])
cib_id_to_idx = {did: i for i, did in enumerate(cib["driver_ids"])}
driver_by_id = {d.id: d for d in drivers}
manif_lookup = {m.id: m for m in morph_box.all_manifestations}
seeds = consistency["configs"]

print(f"Drivers: {len(drivers)}, Manifestations: {len(morph_box.all_manifestations)}")
print(f"Scenario seeds: {len(seeds)}")

In [ ]:
SYSTEM_PROMPTS = {
    "evolutionary": "You are a pragmatic technology analyst writing grounded future scenarios for spectrum monitoring. Focus on operational realities, incremental changes, and practical constraints.",
    "disruptive": "You are a technology foresight expert writing vivid future scenarios for spectrum monitoring. Explore both the promise and the disruption costs of breakthrough technologies.",
    "cautionary": "You are a critical technology analyst writing scenarios that examine what can go wrong with promising technologies. Focus on deployment barriers, organizational resistance, cost overruns, regulatory lag, and unintended consequences.",
    "wildcard": "You are a scenario planner exploring unlikely but plausible futures for spectrum monitoring. Focus on surprising developments, cascading effects, and second-order consequences.",
}

scenarios: list[Scenario] = []
generated_titles: list[str] = []
used_chunk_ids: set[str] = set()

for i, seed in enumerate(seeds):
    stype_str = seed.get("scenario_type", "evolutionary")
    stype = ScenarioType(stype_str) if stype_str in [e.value for e in ScenarioType] else ScenarioType.EVOLUTIONARY
    config = seed["configuration"]

    print(f"\n{'='*60}")
    print(f"Seed {i+1}/{len(seeds)}: {stype.value.upper()} (consistency={seed['consistency_score']:.2f})")
    print(f"{'='*60}")

    # Build driver assumptions from manifestation configuration
    assumptions = []
    manif_block_parts = []
    for d_id, m_id in config.items():
        driver = driver_by_id.get(d_id)
        manif = manif_lookup.get(m_id)
        if not driver or not manif:
            continue
        assumptions.append(DriverAssumption(
            driver_id=d_id,
            manifestation_id=m_id,
            state=manif.label,
            description=f"{driver.name}: {manif.label} — {manif.description}",
        ))
        manif_block_parts.append(
            f"- {driver.name}: **{manif.label}**\n  {manif.description}"
        )
        print(f"  {driver.name[:40]:<41} → {manif.label}")

    driver_manifestations_block = "\n".join(manif_block_parts)

    # CIB context — strongest relationships between this seed's drivers
    cib_parts = []
    seed_driver_ids = list(config.keys())
    for did_a in seed_driver_ids:
        idx_a = cib_id_to_idx.get(did_a)
        if idx_a is None:
            continue
        da = driver_by_id.get(did_a)
        for did_b in seed_driver_ids:
            if did_a == did_b:
                continue
            idx_b = cib_id_to_idx.get(did_b)
            if idx_b is None:
                continue
            score = int(cib_matrix[idx_a][idx_b])
            if abs(score) >= 1:
                db = driver_by_id.get(did_b)
                effect = {2: "strongly promotes", 1: "mildly promotes", -1: "mildly inhibits"}.get(score, "strongly inhibits" if score < -1 else "strongly promotes")
                cib_parts.append(f"- {da.name[:40]} {effect} {db.name[:40]} (score: {score:+d})")
    cib_context = "\n".join(cib_parts) if cib_parts else "No notable cross-impacts."

    existing_titles_block = ""
    if generated_titles:
        existing_titles_block = f"Previously generated scenario titles (yours must be clearly distinct):\n" + "\n".join(f"- {t}" for t in generated_titles)

    # State-aware RAG retrieval
    query_parts = [f"{manif_lookup[config[d_id]].label} {driver_by_id[d_id].name[:30]}"
                   for d_id in list(config.keys())[:4] if d_id in driver_by_id and config[d_id] in manif_lookup]
    query_text = f"{' '.join(query_parts)} spectrum monitoring 2035"
    query_emb = embed([query_text[:500]])[0]
    rag = collection.query(query_embeddings=[query_emb], n_results=8, include=["documents", "metadatas"])

    novel = [(rag["ids"][0][j], rag["documents"][0][j], rag["metadatas"][0][j])
             for j in range(len(rag["ids"][0])) if rag["ids"][0][j] not in used_chunk_ids]
    reused = [(rag["ids"][0][j], rag["documents"][0][j], rag["metadatas"][0][j])
              for j in range(len(rag["ids"][0])) if rag["ids"][0][j] in used_chunk_ids]
    ranked = (novel + reused)[:5]

    rag_text = "\n\n---\n\n".join([
        f"[Chunk ID: {cid}] (Source: {meta['source_title']})\n{doc}"
        for cid, doc, meta in ranked
    ])
    rag_chunk_ids = [cid for cid, _, _ in ranked]

    narrative_guide = prompts.SCENARIO_NARRATIVE_GUIDE.get(
        stype.value, prompts.SCENARIO_NARRATIVE_GUIDE["evolutionary"]
    )

    prompt = prompts.SCENARIO_GENERATE_MORPHOLOGICAL.format(
        driver_manifestations_block=driver_manifestations_block,
        scenario_type=stype.value,
        existing_titles_block=existing_titles_block,
        cib_context=cib_context,
        rag_chunks=rag_text,
        narrative_guide=narrative_guide,
    )

    system = SYSTEM_PROMPTS.get(stype.value, SYSTEM_PROMPTS["evolutionary"])
    result = safe_chat_json(prompt, system=system, temperature=0.7)

    all_source_ids = list(set(rag_chunk_ids + result.get("source_chunk_ids_used", [])))
    used_chunk_ids.update(rag_chunk_ids)

    # Collect source_chunk_ids from manifestations for traceability
    for a in assumptions:
        m = manif_lookup.get(a.manifestation_id)
        if m:
            all_source_ids.extend(m.source_chunk_ids)
    all_source_ids = list(set(all_source_ids))

    scenario = Scenario(
        title=result.get("title", f"Scenario {i+1}"),
        narrative=result.get("narrative", ""),
        type=stype,
        perspective=result.get("perspective", ""),
        key_tensions=result.get("key_tensions", []),
        assumptions=assumptions,
        source_chunk_ids=all_source_ids,
    )
    scenarios.append(scenario)
    generated_titles.append(scenario.title)

    print(f"\n  Title: {scenario.title}")
    print(f"  Perspective: {scenario.perspective}")
    print(f"  Sources: {len(scenario.source_chunk_ids)} chunks")

print(f"\n{'='*60}")
print(f"Generated {len(scenarios)} scenarios from morphological analysis")

# Diversity check
print(f"\nScenario types:")
for s in scenarios:
    manif_count = len([a for a in s.assumptions if a.manifestation_id])
    print(f"  [{s.type.value:14s}] {s.title[:55]} ({manif_count} manifestations)")

In [ ]:
# save scenarios
scenario_state = {
    "scenarios": [s.model_dump(mode="json") for s in scenarios],
}
with open("../data/outputs/scenario_state.json", "w") as f:
    json.dump(scenario_state, f, indent=2)
print(f"Saved {len(scenarios)} scenarios to scenario_state.json")

# chunk overlap check
print(f"\nChunk overlap:")
all_chunks = [set(s.source_chunk_ids) for s in scenarios]
for i in range(len(all_chunks)):
    for j in range(i+1, len(all_chunks)):
        overlap = len(all_chunks[i] & all_chunks[j])
        total = len(all_chunks[i] | all_chunks[j])
        pct = overlap/total*100 if total else 0
        flag = " ⚠" if pct > 50 else ""
        print(f"  S{i+1} vs S{j+1}: {pct:.0f}%{flag}")